# RoadWatch — train on the selected Kaggle dataset
Select Runtime → Change runtime type → GPU. Run cells in order. Have prepare_data.py and train.py ready to upload. No trained weights or measured scores are bundled. GPU availability and completion within two hours are not guaranteed.

In [ ]:
%pip -q install 'ultralytics>=8.3,<9' 'kagglehub>=0.3,<1' 'Pillow>=10,<13' 'PyYAML>=6,<7'
import torch
assert torch.cuda.is_available(), 'Enable a GPU runtime; CPU training may miss the deadline.'
print(torch.cuda.get_device_name(0))

Upload prepare_data.py and train.py together in the next cell.

In [ ]:
from google.colab import files
from pathlib import Path
files.upload()
assert all(Path(p).is_file() for p in ['prepare_data.py','train.py']), 'Upload both scripts.'

Download the selected public Kaggle dataset. If automatic download fails, download its ZIP from https://www.kaggle.com/datasets/chitholian/annotated-potholes-dataset and upload it when prompted. Do not paste API credentials here.

In [ ]:
import kagglehub, zipfile
try:
    source = Path(kagglehub.dataset_download('chitholian/annotated-potholes-dataset'))
except Exception as exc:
    print('Download failed:', type(exc).__name__, 'Upload the dataset ZIP.')
    uploaded = files.upload()
    zips = [p for p in uploaded if p.lower().endswith('.zip')]
    assert len(zips) == 1, 'Upload exactly one ZIP.'
    source = Path('/content/raw').resolve()
    source.mkdir(exist_ok=True)
    with zipfile.ZipFile(zips[0]) as archive:
        for member in archive.infolist():
            assert (source/member.filename).resolve().is_relative_to(source), 'Unsafe archive path'
        archive.extractall(source)
print('Dataset:', source)

In [ ]:
from prepare_data import prepare
dataset = Path('/content/pothole_dataset')
data_yaml = prepare(source, dataset)
print((dataset/'summary.json').read_text())

Fine-tune YOLOv8n: up to 30 epochs, 512-pixel images, batch 8, training time budget 0.75 hours. Setup and test evaluation take additional time. Reduce batch to 4 if GPU memory is insufficient. The validation-selected model is evaluated on held-out test data. Do not tune against test results.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable,'train.py','--data',str(data_yaml),'--epochs','30','--imgsz','512','--batch','8','--device','0','--hours','0.75'],check=True)
print(Path('models/metrics.json').read_text())

In [ ]:
from ultralytics import YOLO
from PIL import Image
from IPython.display import display
model = YOLO('models/best.pt')
sample = sorted((dataset/'images/test').glob('*.jpg'))[0]
result = model.predict(str(sample),conf=0.35,verbose=False)[0]
annotated = Image.fromarray(result.plot()[...,::-1])
annotated.save('test_prediction.png')
display(annotated)

Download actual trained artifacts below and extract into the local project folder. This places best.pt and metrics.json inside models/. Then run setup_windows.bat and run_app.bat. Inspect false alarms and misses before presenting; one image is not sufficient evidence of accuracy.

In [ ]:
import json, importlib.metadata as metadata
Path('environment_versions.json').write_text(json.dumps({p:metadata.version(p) for p in ['ultralytics','torch','Pillow','PyYAML','kagglehub']},indent=2))
artifact = '/content/roadwatch-trained-artifacts.zip'
with zipfile.ZipFile(artifact,'w',zipfile.ZIP_DEFLATED) as archive:
    for path in Path('models').rglob('*'):
        if path.is_file(): archive.write(path,path.as_posix())
    archive.write('test_prediction.png')
    archive.write('environment_versions.json')
    archive.write(dataset/'summary.json','dataset_summary.json')
    archive.write(dataset/'manifest.json','dataset_manifest.json')
    for path in Path('runs').rglob('*'):
        if path.is_file() and path.suffix in ['.png','.csv','.yaml','.json']: archive.write(path,path.as_posix())
files.download(artifact)